# Agents at Scale: Multi-Agent Architecture with A2A Protocol on Agent Runtime and ADK Integration

## Overview

As AI agents take on more responsibilities, a single agent doing everything becomes hard to maintain, scale, and evolve. Different capabilities often need different deployment strategies, update cycles, or even different teams owning them.

* The [A2A (Agent2Agent) Protocol](https://a2a-protocol.org/latest/) solves the communication side — standardizing how agents discover each other's capabilities and collaborate across frameworks and organizations.
* [Gemini Enterprise Agent Platform Runtime](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/overview) solves the deployment side — a fully managed, serverless platform that hosts your agents with built-in A2A support, auto-scaling, secure endpoints, persistent sessions, and zero infrastructure management.

Together, they let you build specialized agents, deploy them as discoverable A2A services, and compose them into multi-agent systems.

## What You'll Build

An **Inventory Agent** that connects to an Inventory MCP server deployed on Google Cloud Run (`inventory-mcp-server` from `mcp_tool_agent.ipynb`). You deploy this agent to Gemini Enterprise Agent Platform Runtime as an A2A service. Then you build a **Store Manager Agent** (`store_manager_agent`) that consumes the deployed Inventory Agent as a remote A2A sub-agent. The result is an enterprise multi-agent architecture where the Store Manager Agent handles general store inquiries and routes inventory queries and stock updates to the remote A2A Inventory Agent.

## Learning Objectives

* Verify Cloud Run deployment of the MCP server built in `mcp_tool_agent.ipynb`.
* Build an ADK agent using `McpToolset` connected to the remote Cloud Run MCP server.
* Expose the MCP-powered agent as an A2A server with agent cards and skills.
* Deploy an A2A agent to Gemini Enterprise Agent Runtime.
* Consume a remote A2A agent from another ADK agent using `RemoteA2aAgent`.
* Test multi-agent systems incrementally using local A2A and ADK Web UI.

> [!IMPORTANT]
> **Prerequisite:** This notebook depends on the Cloud Run MCP server deployed in [`mcp_tool_agent.ipynb`](mcp_tool_agent.ipynb). Please ensure you have executed `mcp_tool_agent.ipynb` to deploy the `inventory-mcp-server` service before running this notebook.


## Environment Setup

We import required packages and set up our Google Cloud project and location environment variables..

In [ ]:
import asyncio
import logging
import os
import sys
import uuid
import warnings
import subprocess
import importlib

import google.auth
import vertexai

from a2a.helpers import new_text_message
from a2a.server.context import ServerCallContext
from a2a.types import (
    AgentCard,
    AgentSkill,
    GetExtendedAgentCardRequest,
    GetTaskRequest,
    Message,
    Part,
    Role,
    SendMessageRequest,
    TaskState,
)
from google.adk.a2a.utils.agent_to_a2a import (
    A2aAgentExecutor,
    AgentCardBuilder,
    to_a2a,
)
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.auth import default
from google.genai import Client, types
from vertexai.agent_engines.templates.a2a import A2aAgent, create_agent_card

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.ERROR)
logging.getLogger("opentelemetry.context").setLevel(logging.CRITICAL)

In [ ]:
# Set Google Cloud Project and Location
_, PROJECT_ID = google.auth.default()
LOCATION = "global"
REGION = "us-central1"
CLOUD_RUN_LOCATION = "us-central1"
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["REGION"] = REGION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"  # Use Agent Platform API

### Verify Deployed Cloud Run MCP Server

Before building the A2A agent, we check that the `inventory-mcp-server` Cloud Run service deployed in [`mcp_tool_agent.ipynb`](mcp_tool_agent.ipynb) is active and retrieve its service URL. If the service is not found, an error will be raised.

In [ ]:
# Create directory structure for our consumer agent files
!mkdir -p ./a2a_agent/store_manager_agent

cmd = [
    "gcloud", "run", "services", "describe", "inventory-mcp-server",
    "--region", CLOUD_RUN_LOCATION,
    "--project", PROJECT_ID,
    "--format", "value(status.url)"
]

try:
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    cloud_run_url = result.stdout.strip()
    if not cloud_run_url:
        raise RuntimeError("Cloud Run service URL returned empty.")
    MCP_SERVER_URL = f"{cloud_run_url}/sse"
    os.environ["REMOTE_MCP_SERVER_URL"] = MCP_SERVER_URL
    print(f"Verified Cloud Run MCP Server URL: {MCP_SERVER_URL}")
except Exception as e:
    raise RuntimeError(
        "Could not find deployed 'inventory-mcp-server' Cloud Run service. "
        "Please execute 'mcp_tool_agent.ipynb' first to deploy the Cloud Run MCP server."
    ) from e

## Concept: Agent2Agent (A2A) Protocol and Gemini Enterprise Agent Runtime

### The Agent2Agent (A2A) Protocol
The [Agent2Agent (A2A) protocol](https://a2a-protocol.org) is an open standard designed to enable seamless communication and collaboration between AI agents. Where MCP (Model Context Protocol) connects agents to *tools and data*, A2A connects agents to *other agents* — enabling them to discover each other's capabilities, delegate tasks, and collaborate across frameworks and organizations.

The key difference between wrapping an agent as a tool (via MCP) vs exposing it via A2A: tools are stateless and perform single functions, while A2A agents can reason, maintain state, and handle multi-turn interactions like negotiation or clarification. An agent exposed via A2A retains its full capabilities rather than being reduced to a function call.

A2A defines three core concepts:
1. **Agent Card** — a JSON document describing what an agent does, its skills, and its endpoint. Other agents fetch this card to discover capabilities.
2. **Message** — a user or agent request sent to an A2A endpoint, triggering a task.
3. **Task** — a unit of work with a lifecycle (submitted → working → completed/failed) and **artifacts** containing the results.

### Gemini Enterprise Agent Platform Runtime
**Agent Runtime** is a fully managed service on Google Cloud for deploying, scaling, and managing AI agents in production with Enterprise security features (e.g. VPC Service Controls, CMEK). It handles infrastructure so you can focus on agent logic.

Agent Runtime provides:
* **Managed deployment** — deploy agents built with ADK, LangGraph, or any Python framework with a single SDK call
* **A2A hosting** — deploy agents as A2A-compliant endpoints with automatic agent card serving and authenticated access
* **Persistent sessions** — `VertexAiSessionService` stores conversation history and state across requests
* **Auto-scaling** — scales from zero to handle traffic, with no infrastructure management
* **Observability** — built-in tracing, logging, and monitoring via Google Cloud's observability stack

In this tutorial, you deploy the inventory agent to Agent Runtime. The deployment process serializes (pickles) your agent code and uploads it. Agent Runtime provisions a serverless endpoint that serves the A2A protocol — other agents (or clients) interact with it via standard HTTP calls, authenticated with Google Cloud credentials.

## Exposing the Inventory Agent via A2A Protocol

### Import Existing Inventory MCP Agent (`custom_mcp_agent.agent.agent`)

We import `root_agent` directly from `custom_mcp_agent/agent/agent.py` created in `mcp_tool_agent.ipynb`. This agent uses `McpToolset` connected to our Cloud Run MCP server.

In [ ]:
if "." not in sys.path:
    sys.path.insert(0, ".")

import custom_mcp_agent.agent.agent
importlib.reload(custom_mcp_agent.agent.agent)
from custom_mcp_agent.agent.agent import root_agent

print(f"Loaded ADK Agent: {root_agent.name}")
print(f"Description: {root_agent.description}")
print(f"MCP Server URL: {custom_mcp_agent.agent.agent.MCP_SERVER_URL}")

### Convert to an A2A ASGI Application (`to_a2a`) for Local Testing

ADK provides a built-in utility function `to_a2a(root_agent)` that converts any ADK agent into an A2A-compliant ASGI Starlette application without requiring a custom configuration file or custom executor boilerplate.

### Why `to_a2a()`?
1. **Zero Boilerplate**: Automatically creates an under-the-hood `A2aAgentExecutor` backed by ADK runners and in-memory services.
2. **Automatic Agent Card Generation**: Extracts the agent's name, description, and tools into a standard `.well-known/agent-card.json` card automatically.
3. **Local Serving & Debugging**: Can be served immediately via `uvicorn agent:a2a_app --port 8001` or CLI (`adk api_server --a2a`) and tested with `adk web`.

In [ ]:
from custom_mcp_agent.agent.agent import root_agent

# 1. Convert the ADK agent directly to an A2A ASGI Starlette application
#    In a standalone server, you can serve this app using:
#    uvicorn custom_mcp_agent.agent.agent:a2a_app --host localhost --port 8001
a2a_app = to_a2a(root_agent, host="localhost", port=8001, protocol="http")

# 2. auto-generate Agent Card using AgentCardBuilder
card_builder = AgentCardBuilder(
    agent=root_agent, rpc_url="http://localhost:8001/"
)
auto_card = await card_builder.build()

print("=== ADK Auto-Generated Agent Card ===")
print(f"Name: {auto_card.name}")
print(f"Description: {auto_card.description}")
print(f"Skills: {[skill.name for skill in auto_card.skills]}")
print(
    f"Protocol Binding: {auto_card.supported_interfaces[0].protocol_binding}"
)
print(f"RPC URL: {auto_card.supported_interfaces[0].url}")

### Prepare the A2A Agent for Cloud Deployment (`A2aAgent` + `A2aAgentExecutor`) and Test Locally

While `to_a2a()` creates an ASGI application for self-hosting (such as on Cloud Run or a local container), **Google Cloud Agent Runtime** (`client.agent_engines.create(...)`) expects the `vertexai.agent_engines.templates.a2a.A2aAgent` template class with an `HTTP+JSON` interface binding.

Instead of writing a custom executor from scratch, we use ADK's built-in **`A2aAgentExecutor`** (`from google.adk.a2a.utils.agent_to_a2a import A2aAgentExecutor`) as the executor builder for `A2aAgent`.

In this step, we wrap our inventory agent using `A2aAgent` + `A2aAgentExecutor` and test the complete A2A protocol lifecycle locally inside the notebook:
1. **Agent Card Retrieval** (`on_get_extended_agent_card`) — discovering capabilities and skills.
2. **Message Sending** (`on_message_send`) — submitting a request to create a task.
3. **Task Status & Artifacts** (`on_get_task`) — polling for task completion and extracting results.
4. **Session Continuity** — reusing `contextId` across requests so the agent remembers previous turns.

#### Define Helper Functions for Mock A2A Requests

In standard A2A deployments, the Agent Runtime server converts incoming HTTP requests into Starlette `Request` objects. To test our `A2aAgent` locally without spinning up an external web server, we define helper functions (`build_get_request`, `build_post_request`, `wait_for_task`) that simulate incoming HTTP requests and poll task status.

In [ ]:
# Helper functions to test A2A protocol methods locally using SDK protobuf types
def create_call_context() -> ServerCallContext:
    """Creates a default ServerCallContext for local method invocation."""
    return ServerCallContext()


def build_send_message_request(text: str, context_id: str = "") -> SendMessageRequest:
    """Builds a SendMessageRequest with a user text message."""
    return SendMessageRequest(
        message=Message(
            message_id=f"msg-{uuid.uuid4().hex[:8]}",
            role=Role.ROLE_USER,
            parts=[Part(text=text)],
            context_id=context_id,
        )
    )


async def wait_for_task(a2a_agent, task_id: str, context: ServerCallContext, max_retries: int = 30):
    """Poll on_get_task until the task reaches completed or failed state."""
    for _ in range(max_retries):
        result = await a2a_agent.on_get_task(
            request=GetTaskRequest(id=task_id),
            context=context,
        )
        if result and result.status.state in [
            TaskState.TASK_STATE_COMPLETED,
            TaskState.TASK_STATE_FAILED,
        ]:
            return result
        await asyncio.sleep(1)
    return result


def print_task_answer(result):
    """Extract and print the text response from task artifacts."""
    if not result:
        print("No task result returned.")
        return
    print(f"Status: {TaskState.Name(result.status.state)}")
    for artifact in result.artifacts:
        for part in artifact.parts:
            if part.text:
                print(f"Answer: {part.text}")

#### Initialize the Local A2A Agent

We instantiate `A2aAgent` passing our `agent_card` and ADK's built-in `A2aAgentExecutor`, then call `set_up()` to initialize the ADK runner and in-memory session service.

In [ ]:
# Ensure current working directory is in sys.path to import modules
import sys
import importlib

if "." not in sys.path:
    sys.path.insert(0, ".")

import custom_mcp_agent.agent.agent
importlib.reload(custom_mcp_agent.agent.agent)
from custom_mcp_agent.agent.agent import root_agent

# 1. Prepare a function builder for the ADK Runner so the agent template remains picklable for deployment
def get_inventory_runner():
    import importlib
    import custom_mcp_agent.agent.agent
    importlib.reload(custom_mcp_agent.agent.agent)
    from custom_mcp_agent.agent.agent import root_agent
    return Runner(
        app_name=root_agent.name,
        agent=root_agent,
        session_service=InMemorySessionService(),
    )

# 2. Build the AgentCard for Google Cloud Agent Runtime (requires HTTP+JSON binding)
skills = [
    AgentSkill(
        id="inventory_management",
        name="inventory_management",
        description="Query inventory items (list SKUs) and update stock quantities.",
        tags=["inventory", "mcp"],
        input_modes=["text/plain"],
        output_modes=["text/plain"],
    )
]
agent_card = create_agent_card(
    agent_name=root_agent.name,
    description=root_agent.description,
    skills=skills,
)

# 3. Initialize A2aAgent using ADK's built-in A2aAgentExecutor
a2a_agent = A2aAgent(
    agent_card=agent_card,
    agent_executor_builder=A2aAgentExecutor,
    agent_executor_kwargs={"runner": get_inventory_runner},
    extended_agent_card=agent_card,
)
a2a_agent.set_up()
print("=== A2A Agent Successfully Initialized ===")
print(f"Name: {a2a_agent.agent_card.name}")
print(f"Skills: {[skill.name for skill in a2a_agent.agent_card.skills]}")

#### Test 1: Discovering the Agent Card

Other agents in an A2A ecosystem discover an agent's name, description, and skills by querying its agent card endpoint (`/v1/card`). We can test calling `on_get_extended_agent_card`:

In [ ]:
# 1. Test Agent Card Retrieval
card_response = await a2a_agent.on_get_extended_agent_card(
    request=GetExtendedAgentCardRequest(),
    context=create_call_context(),
)

print("=== Agent Card ===")
print(f"Name: {card_response.name}")
print(f"Description: {card_response.description}")
print(f"Skills: {[s.name for s in card_response.skills]}")


#### Test 2: Querying Inventory SKUs

In the A2A protocol, sending a message to an agent creates or continues a conversation task. Let's send a request to list available SKUs in inventory.

In [ ]:
# 2. Test Listing SKUs via A2A Message
request = build_send_message_request(
    "List all available SKUs in the inventory."
)
context = create_call_context()
response = await a2a_agent.on_message_send(request=request, context=context)

task_id = response.id
context_id = response.context_id
print(f"Task submitted! Task ID: {task_id}")
print(f"Session Context ID: {context_id}")

# Poll until the task completes
result = await wait_for_task(a2a_agent, task_id, context)
print("\n=== Inventory Query Response ===")
print_task_answer(result)

#### Test 3: Updating Inventory Quantity (Session Continuity)

To test multi-turn conversation and session continuity over A2A, pass the `context_id` returned from the previous request. Let's update the stock quantity of a SKU.

In [ ]:
# 3. Test Updating SKU Quantity using contextId for session continuity
request = build_send_message_request(
    "Update quantity for SKU001 to 50 items",
    context_id=context_id,
)
update_response = await a2a_agent.on_message_send(
    request=request, context=context
)

update_result = await wait_for_task(a2a_agent, update_response.id, context)
print("=== Update Inventory Response ===")
print_task_answer(update_result)

#### Test 4: Verifying Inventory Update

Finally, let's verify that the updated quantity is reflected in subsequent queries over the A2A protocol.

In [ ]:
# 4. Test Verifying Inventory Update via A2A Message
request = build_send_message_request(
    "List all SKUs again to confirm the updated quantity for SKU001.",
    context_id=context_id,
)
verify_response = await a2a_agent.on_message_send(
    request=request, context=context
)

verify_result = await wait_for_task(a2a_agent, verify_response.id, context)
print("=== Verify Inventory Response ===")
print_task_answer(verify_result)

### Deploy the Inventory Agent to Agent Runtime

In this step, you deploy your inventory agent to Gemini Enterprise Agent Platform Runtime. Agent Runtime packages your agent into a container, deploys it to a serverless endpoint, and handles A2A protocol routing.

#### Deploying via `client.agent_engines.create(...)`

We use the Vertex AI Python SDK to deploy our `A2aAgent`. Because creating Google Cloud resources takes 3-5 minutes, the deployment block below is controlled by a flag `DEPLOY_TO_AGENT_RUNTIME = False`. Set it to `True` if you wish to deploy:

In [ ]:
RESOURCE_NAME = os.environ.get("INVENTORY_AGENT_RESOURCE_NAME", "")

print("Starting deployment to Agent Runtime (this takes 3-5 minutes)...")
STAGING_BUCKET = os.environ.get(
    "STAGING_BUCKET", f"{PROJECT_ID}-adk-a2a-agent-runtime"
)
BUCKET_URI = f"gs://{STAGING_BUCKET}"

vertexai.init(
    project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI
)
client = vertexai.Client(
    project=PROJECT_ID,
    location=REGION,
    http_options=types.HttpOptions(api_version="v1beta1"),
)

deploy_a2a_agent = A2aAgent(
    agent_card=agent_card,
    agent_executor_builder=A2aAgentExecutor,
    agent_executor_kwargs={"runner": get_inventory_runner},
    extended_agent_card=agent_card,
)

remote_agent = client.agent_engines.create(
    agent=deploy_a2a_agent,
    config={
        "display_name": agent_card.name,
        "description": agent_card.description,
        "requirements": [
            "google-cloud-aiplatform[agent_engines,adk]==1.160.0",
            "google-genai==2.11.0",
            "google-adk[mcp, a2a]==2.5.0",
            "mcp>=1.25.0,<2.0.0",
            "a2a-sdk==1.1.2",
            "cloudpickle",
            "pydantic",
        ],
        "extra_packages": [
            "./custom_mcp_agent",
        ],
        "http_options": {
            "api_version": "v1beta1",
        },
        "staging_bucket": BUCKET_URI,
    },
)

RESOURCE_NAME = remote_agent.api_resource.name
os.environ["INVENTORY_AGENT_RESOURCE_NAME"] = RESOURCE_NAME
print(f"\nDeployment complete! Resource name: {RESOURCE_NAME}")

#### Testing the Remote Deployed A2A Agent

Once deployed, any authorized client can interact with the agent over HTTP. We demonstrate fetching the remote agent card and sending an inventory request using `client.agent_engines.get(...)`:

In [ ]:
# 1. Fetch Remote Agent Card
card = await remote_agent.on_get_extended_agent_card(
    request=GetExtendedAgentCardRequest()
)
print(f"Connected to Remote Agent: {card.name}")

# 2. Send Message to Remote Agent
print("\nSending inventory request...")
req = SendMessageRequest(
    message=new_text_message(
        "List all SKUs available in inventory."
    )
)
response = await remote_agent.on_message_send(request=req)

# 3. Print Task Response
for item in response:
    task = item.task
    if hasattr(task, "artifacts") and task.artifacts:
        for art in task.artifacts:
            for part in art.parts:
                text = getattr(part, "text", None) or (
                    part.root.text if hasattr(part, "root") else None
                )
                if text:
                    print(f"\nResponse:\n{text}")

## Consuming A2A Agent


### Integrate A2A Inventory Agent with Store Manager Agent

In this step, we create the **Store Manager Agent** (`store_manager_agent`) to consume our inventory agent as a remote A2A sub-agent using ADK's `RemoteA2aAgent`.

When integrated:
* General store inquiries ("What are the store hours?") are handled by local store tools.
* Inventory stock queries and updates ("Check stock of Wireless Headphones" or "Update SKU001 quantity to 50") are automatically delegated to our `RemoteA2aAgent` over the A2A protocol.

#### Resolve the Agent Card URL

To delegate tasks via A2A, `RemoteA2aAgent` requires the full URL to the remote agent card (`{card.url}/v1/card`). We can resolve it dynamically from `RESOURCE_NAME` (or set a default card URL for testing):

In [ ]:
card = await remote_agent.on_get_extended_agent_card(
    request=GetExtendedAgentCardRequest()
)
# Point the supported interface URL to our deployed Agent Runtime endpoint
card.supported_interfaces[0].url = f"https://{REGION}-aiplatform.googleapis.com/v1beta1/{RESOURCE_NAME}/a2a"
print(f"Connected to Remote Agent: {card.name}")
print(f"Resolved Endpoint URL: {card.supported_interfaces[0].url}")

INVENTORY_AGENT_CARD_URL = f"{card.supported_interfaces[0].url}"
os.environ["INVENTORY_AGENT_CARD_URL"] = INVENTORY_AGENT_CARD_URL

#### Create the Store Manager Agent (`store_manager_agent/agent.py`)

We write `./a2a_agent/store_manager_agent/agent.py` to define the root consumer agent (`store_manager_agent`). In this file, we configure:

1. **Local Store Tools (`get_store_info`)**: Local Python functions for handling general store inquiries like operating hours, contact info, and location.
2. **Authenticated HTTP Client (`get_gcp_httpx_client`)**: Uses Google Application Default Credentials (`google.auth.default()`) to generate OAuth bearer tokens, ensuring secure communication with the remote A2A Agent Runtime service.
3. **Remote A2A Sub-Agent (`RemoteA2aAgent`)**: Wraps the remote A2A `inventory_assistant` by specifying its `AgentCard` and HTTP+JSON endpoint URL (`INVENTORY_AGENT_CARD_URL`).
4. **Orchestrator Agent (`root_agent`)**: An `LlmAgent` powered by Gemini 3.5 Flash equipped with `get_store_info` and `AgentTool(inventory_remote_agent)`. When a user asks about inventory or stock, the Store Manager automatically delegates the query to the remote `inventory_assistant` via AgentTool over the A2A protocol.

In [ ]:
%%writefile ./a2a_agent/store_manager_agent/__init__.py
# Package init for store_manager_agent

In [ ]:
%%writefile ./a2a_agent/store_manager_agent/agent.py
# store_manager_agent/agent.py
"""Store Manager Agent module for A2A integration."""

import os

import httpx
from a2a.types import AgentCard, AgentInterface
from google.adk.agents import LlmAgent
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools import AgentTool
from google.auth import default
from google.auth.transport.requests import Request as AuthRequest

INVENTORY_AGENT_CARD_URL = os.environ.get("INVENTORY_AGENT_CARD_URL", "")


def get_store_info(info_type: str = "hours") -> str:
    """Get general store information like operating hours or contact details."""
    if "hour" in str(info_type).lower():
        return (
            "Store hours: Monday-Saturday 8:00 AM - 9:00 PM, "
            "Sunday 10:00 AM - 6:00 PM."
        )
    if "contact" in str(info_type).lower():
        return (
            "Store contact: support@store.example.com, Phone: (555) 019-2834."
        )
    return "Store location: 123 Main Street, Suite 100."


def get_gcp_httpx_client(timeout: int = 60) -> httpx.AsyncClient:
    """Create an authenticated HTTP client using ADC."""
    scopes = ["https://www.googleapis.com/auth/cloud-platform"]
    credentials, _ = default(scopes=scopes)
    if not credentials.valid:
        credentials.refresh(AuthRequest())
    return httpx.AsyncClient(
        headers={"Authorization": f"Bearer {credentials.token}"},
        timeout=timeout,
    )


# Define remote A2A sub-agent if URL is configured
agent_card = AgentCard(
    name="inventory_assistant",
    description=(
        "Specialized assistant for inventory management, listing SKUs, "
        "and updating stock quantities."
    ),
    supported_interfaces=[
        AgentInterface(
            url=INVENTORY_AGENT_CARD_URL,
            protocol_binding="HTTP+JSON",
            protocol_version="1.0",
        )
    ],
    skills=[],
)

inventory_remote_agent = RemoteA2aAgent(
    name="inventory_assistant",
    description=(
        "Specialized assistant for inventory management, listing SKUs, "
        "and updating stock quantities. Delegate to this agent for any "
        "inventory or stock operations."
    ),
    agent_card=agent_card,
    httpx_client=get_gcp_httpx_client(),
)


root_agent = LlmAgent(
    name="store_manager_agent",
    model=Gemini(
        model="gemini-3.5-flash",
        client_kwargs={"vertexai": True, "location": "global"},
    ),
    instruction="""You are a Store Manager Assistant responsible for overall store operations. Your job:
- Provide store information such as opening hours, contact details, and store policies using store tools.
- For any inventory questions, listing SKUs, checking item stock, or updating item quantities, delegate to the inventory_assistant.

Be conversational, helpful, and concise.""",
    tools=[get_store_info, AgentTool(inventory_remote_agent)],
)

In [ ]:
%%writefile ./a2a_agent/store_manager_agent/.env
# Specify global location for Gemini models and Vertex AI ADC
GOOGLE_CLOUD_LOCATION=global
GOOGLE_GENAI_USE_VERTEXAI=TRUE

#### Test the Integrated Agent
You can test the integrated store manager agent using the ADK Developer UI (`adk web`).

In [ ]:
# On Cloud Workstations
!adk web a2a_agent/store_manager_agent --allow_origins "regex:https://.*\.cloudworkstations\.dev"

**Note**: If you are using Agent Platform Workbench, remove the comment out and run the cell below.



In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# echo "--------------------------------------------------------"
# echo "🔗 ACCESS HERE: https://${PROXY_BASE}/proxy/8000"
# echo "--------------------------------------------------------"
# adk web . --url_prefix /proxy/8000  --allow_origins "regex:https://.*\.notebooks\.googleusercontent\.com"


Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.